# jaehyuk_pipeline — 공통 fold·홀드아웃 확정

수아 `suah_feature_exp.ipynb`의 STEP 2(사상 ID)·3(홀드아웃)·6(FOLDS)과 **같은 방식**으로 만들어서 파일로 고정한다.
→ 수아가 이미 돌린 1차 실험 결과(`outputs/feature_exp_v1/`)와 분할이 같게 유지된다.

산출물 (`outputs/common/`, git에 올림)
- `event_folds.csv`: `event_id, fold, n_rows` (홀드아웃은 fold = -1). fold는 사상 단위라 사상→fold 매핑만 있으면 충분
- `holdout_events.json`: 홀드아웃 사상 목록

**한 번만 실행하고 결과 파일을 공유한다. 다시 만들지 않는다.**

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_DIR = ROOT / "data"
OUT_DIR = DATA_DIR.parent / "outputs"
COMMON_DIR = OUT_DIR / "common"
COMMON_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_FOLDS = 4
HOLDOUT_FRAC = 0.15
TARGET = "target_maxrise_6h"
EXPECTED = {"train_rows": 749_657, "train_events": 122}

## STEP 1. 로드 + 사상 ID 복원 (베이스라인·수아 노트북과 동일)

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv.gz")
train["timestamp"] = pd.to_datetime(train["timestamp"])

def add_event_id(df, prefix, gap_h=1):
    ts = pd.Series(np.sort(df["timestamp"].unique()))
    event_no = ts.diff().gt(pd.Timedelta(hours=gap_h)).cumsum()
    m = pd.DataFrame({"timestamp": ts, "event_id": prefix + event_no.astype(str).str.zfill(3)})
    return df.merge(m, on="timestamp", how="left")

train = add_event_id(train, "TR_E")

assert len(train) == EXPECTED["train_rows"], f"행 수 다름: {len(train)}"
assert train["row_id"].is_unique, "row_id 중복"
print("사상 수:", train["event_id"].nunique(), "(기대", EXPECTED["train_events"], ")")
assert train["event_id"].nunique() == EXPECTED["train_events"], "사상 수 다름 → gap_h 확인 필요"

## STEP 2. 홀드아웃 (수아 STEP 3과 동일: 정렬된 사상에서 seed 42로 15%)

In [ ]:
events = np.sort(train["event_id"].unique())
rng = np.random.default_rng(SEED)
n_hold = int(round(len(events) * HOLDOUT_FRAC))
hold_events = sorted(rng.choice(events, size=n_hold, replace=False).tolist())
print(f"홀드아웃 {len(hold_events)}개:", hold_events)

# 수아 1차 실험의 홀드아웃과 같은지 확인
suah_path = OUT_DIR / "feature_exp_v1" / "holdout_events.csv"
if suah_path.exists():
    suah_hold = sorted(pd.read_csv(suah_path)["event_id"].tolist())
    assert suah_hold == hold_events, "수아 홀드아웃과 다름 → 팀에 공유 필요"
    print("수아 홀드아웃과 일치 ✅")

## STEP 3. 4-fold (수아 STEP 6과 동일: make_features와 같은 정렬 후 GroupKFold)

In [ ]:
dev = (train[~train["event_id"].isin(hold_events)]
       .sort_values(["station_id", "timestamp"]).reset_index(drop=True))

fold_of = {e: -1 for e in hold_events}
for k, (_, i_va) in enumerate(GroupKFold(n_splits=N_FOLDS).split(dev, dev[TARGET], dev["event_id"])):
    for e in dev.loc[i_va, "event_id"].unique():
        assert e not in fold_of, f"{e}가 여러 fold에 걸침"
        fold_of[e] = k

event_folds = (train.groupby("event_id").size().rename("n_rows").reset_index()
               .assign(fold=lambda d: d["event_id"].map(fold_of))
               [["event_id", "fold", "n_rows"]])
assert event_folds["fold"].notna().all() and len(event_folds) == EXPECTED["train_events"]

## STEP 4. fold별 분포 확인
행 수·target 평균이 fold끼리 크게 다르면 팀에 공유

In [ ]:
chk = train.merge(event_folds[["event_id", "fold"]], on="event_id")
chk.groupby("fold").agg(
    events=("event_id", "nunique"),
    rows=("row_id", "size"),
    target_mean=(TARGET, "mean"),
    target_std=(TARGET, "std"),
    target_max=(TARGET, "max"),
).round(4)

## STEP 5. 저장 (덮어쓰기 방지)

In [ ]:
f_path = COMMON_DIR / "event_folds.csv"
h_path = COMMON_DIR / "holdout_events.json"
assert not f_path.exists(), "이미 있음. 다시 만들 거면 지우고 팀에 알리기"

event_folds.to_csv(f_path, index=False)
with open(h_path, "w", encoding="utf-8") as f:
    json.dump({"seed": SEED, "frac": HOLDOUT_FRAC, "n_folds": N_FOLDS, "events": hold_events},
              f, ensure_ascii=False, indent=2)
print("저장:", f_path, h_path)

## 팀원 사용법

각자 노트북에서 `add_event_id`로 사상 ID를 만든 뒤 아래 코드로 fold를 붙인다. 노트북 안에서 `GroupKFold`·홀드아웃을 새로 만들지 않는다.

```python
ef = pd.read_csv(OUT_DIR / "common" / "event_folds.csv")

# 사상 ID 복원이 같은지 확인 (사상별 행 수 비교)
my_counts = train.groupby("event_id").size()
assert my_counts.reindex(ef["event_id"]).values.tolist() == ef["n_rows"].tolist(), "사상 ID 복원 결과가 다름"

tr = tr.merge(ef[["event_id", "fold"]], on="event_id", how="left")
hold_events = set(ef.loc[ef["fold"] == -1, "event_id"])

dev = tr[tr["fold"] >= 0].reset_index(drop=True)    # 실험용
hold = tr[tr["fold"] == -1].reset_index(drop=True)  # 최종 확정 후 한 번만

FOLDS = [(np.where(dev["fold"] != k)[0], np.where(dev["fold"] == k)[0]) for k in range(4)]
```
`FOLDS` 형식이 기존과 같아서 `run_cv`는 그대로 쓰면 된다.